# API

In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"
)

response = client.chat.completions.create(
    model="default",
    messages=[{"role": "user", "content": "Say hello in 5 words."}]
)

print(response.choices[0].message.content)

Hello! Hope you are wonderful.


# Knowledge Base

In [2]:
documents = [
    "The Aurora Bike Company was founded in 2015 in Portland, Oregon, by two former mechanical engineers who wanted to build lightweight urban bikes.",
    "Aurora bikes are known for their lightweight aluminum frames, which are manufactured using a proprietary welding process developed in-house.",
    "The best-selling model is the Aurora Sprint, released in 2019. It quickly became popular among city commuters for its speed and low weight.",
    "Aurora offers a 2-year warranty on all frames, covering manufacturing defects but not damage from accidents or improper use.",
    "The company's headquarters in Portland has a small museum showing the history of bike design, including early prototypes of the Sprint.",
    "Aurora's electric bike line launched in 2022, called Aurora Volt. It uses a 250W motor and has a range of about 60 kilometers per charge.",
    "Customer support is available Monday to Friday, 9am-5pm PST, through phone, email, and live chat on the Aurora website.",
    "Aurora bikes are sold in over 30 countries worldwide, with major markets in the United States, Germany, and Japan.",
    "The Aurora Sprint weighs only 9.5 kg, making it one of the lightest bikes in its price range.",
    "Aurora's return policy allows returns within 30 days of purchase, provided the bike is unused and in its original packaging.",
    "In 2023, Aurora partnered with a recycling company to offer a bike trade-in program for customers upgrading to newer models.",
    "The Aurora Volt's battery can be fully charged in 3 hours using the included fast charger.",
    "Aurora's design team is based in Portland, while manufacturing takes place in a partner facility in Taiwan.",
    "The company sponsors a yearly urban cycling event called Aurora Ride Day, held every June in Portland.",
    "Aurora's most affordable model, the Aurora Basic, was discontinued in 2021 due to low demand."
]

print(f"Loaded {len(documents)} documents")

Loaded 15 documents


In [3]:
for i, doc in enumerate(documents):
    print(f"doc_{i}: {doc}")

doc_0: The Aurora Bike Company was founded in 2015 in Portland, Oregon, by two former mechanical engineers who wanted to build lightweight urban bikes.
doc_1: Aurora bikes are known for their lightweight aluminum frames, which are manufactured using a proprietary welding process developed in-house.
doc_2: The best-selling model is the Aurora Sprint, released in 2019. It quickly became popular among city commuters for its speed and low weight.
doc_3: Aurora offers a 2-year warranty on all frames, covering manufacturing defects but not damage from accidents or improper use.
doc_4: The company's headquarters in Portland has a small museum showing the history of bike design, including early prototypes of the Sprint.
doc_5: Aurora's electric bike line launched in 2022, called Aurora Volt. It uses a 250W motor and has a range of about 60 kilometers per charge.
doc_6: Customer support is available Monday to Friday, 9am-5pm PST, through phone, email, and live chat on the Aurora website.
doc_7:

# Embeddings

In [4]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load a small, fast, free embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for our documents
embeddings = embed_model.encode(documents).tolist()

print(f"Created {len(embeddings)} embeddings, each of size {len(embeddings[0])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Created 15 embeddings, each of size 384


In [5]:
len(embeddings)

15

In [6]:
chroma_client = chromadb.Client()

# Delete the existing collection
try:
    chroma_client.delete_collection(name="aurora_bikes")
except:
    pass
    
collection = chroma_client.create_collection(name="aurora_bikes")

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("Documents stored in vector DB")

Documents stored in vector DB


# Retrieve 

In [7]:
def retrieve(query, n_results=5):
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return results['documents'][0]

# quick test
test_results = retrieve("How much does the Aurora Sprint weigh?")
for r in test_results:
    print("-", r)

- The Aurora Sprint weighs only 9.5 kg, making it one of the lightest bikes in its price range.
- The best-selling model is the Aurora Sprint, released in 2019. It quickly became popular among city commuters for its speed and low weight.
- Aurora's electric bike line launched in 2022, called Aurora Volt. It uses a 250W motor and has a range of about 60 kilometers per charge.
- Aurora's most affordable model, the Aurora Basic, was discontinued in 2021 due to low demand.
- The Aurora Bike Company was founded in 2015 in Portland, Oregon, by two former mechanical engineers who wanted to build lightweight urban bikes.


# Simple RAG

In [8]:
def rag_answer(query):
    # 1. Retrieve relevant chunks
    context_docs = retrieve(query)
    context = "\n".join(context_docs)

    # 2. Build prompt with context
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know."

Context:
{context}

Question: {query}
Answer:"""

    # 3. Ask the LLM
    response = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# test it
print(rag_answer("How much does the Aurora Sprint weigh?"))

9.5 kg


In [9]:
print(rag_answer("What color is the Aurora Sprint?"))

I don't know.


In [10]:
print(rag_answer("How long does the Volt take to charge?"))

The Aurora Volt's battery can be fully charged in 3 hours using the included fast charger.


# Evaluation Script

In [11]:
test_set = [
    {"question": "How much does the Aurora Sprint weigh?", "expected_doc_id": "doc_8"},
    {"question": "When was Aurora Bike Company founded?", "expected_doc_id": "doc_0"},
    {"question": "How long does the Volt take to charge?", "expected_doc_id": "doc_11"},
    {"question": "What is Aurora's return policy?", "expected_doc_id": "doc_9"},
    {"question": "Where does Aurora manufacture its bikes?", "expected_doc_id": "doc_12"},
]

In [12]:
def evaluate_retrieval(test_set, k=3):
    correct = 0
    for item in test_set:
        query_embedding = embed_model.encode([item["question"]]).tolist()
        results = collection.query(query_embeddings=query_embedding, n_results=k)
        retrieved_ids = results['ids'][0]

        hit = item["expected_doc_id"] in retrieved_ids
        correct += hit
        print(f"Q: {item['question']}")
        print(f"  Expected: {item['expected_doc_id']} | Retrieved: {retrieved_ids} | {'✓' if hit else '✗'}\n")

    accuracy = correct / len(test_set)
    print(f"Recall@{k}: {accuracy:.2f} ({correct}/{len(test_set)})")

evaluate_retrieval(test_set, k=3)

Q: How much does the Aurora Sprint weigh?
  Expected: doc_8 | Retrieved: ['doc_8', 'doc_2', 'doc_5'] | ✓

Q: When was Aurora Bike Company founded?
  Expected: doc_0 | Retrieved: ['doc_0', 'doc_1', 'doc_10'] | ✓

Q: How long does the Volt take to charge?
  Expected: doc_11 | Retrieved: ['doc_11', 'doc_5', 'doc_6'] | ✓

Q: What is Aurora's return policy?
  Expected: doc_9 | Retrieved: ['doc_9', 'doc_3', 'doc_10'] | ✓

Q: Where does Aurora manufacture its bikes?
  Expected: doc_12 | Retrieved: ['doc_0', 'doc_7', 'doc_1'] | ✗

Recall@3: 0.80 (4/5)


# Try a better embedding model

In [13]:
embed_model_v2 = SentenceTransformer('all-mpnet-base-v2')

embeddings_v2 = embed_model_v2.encode(documents).tolist()

chroma_client.delete_collection(name="aurora_bikes_v2") if "aurora_bikes_v2" in [c.name for c in chroma_client.list_collections()] else None

collection_v2 = chroma_client.create_collection(name="aurora_bikes_v2")
collection_v2.add(
    documents=documents,
    embeddings=embeddings_v2,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("v2 collection ready")

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

v2 collection ready


In [14]:
def evaluate_retrieval_v2(test_set, k=3):
    correct = 0
    for item in test_set:
        query_embedding = embed_model_v2.encode([item["question"]]).tolist()
        results = collection_v2.query(query_embeddings=query_embedding, n_results=k)
        retrieved_ids = results['ids'][0]

        hit = item["expected_doc_id"] in retrieved_ids
        correct += hit
        print(f"Q: {item['question']}")
        print(f"  Expected: {item['expected_doc_id']} | Retrieved: {retrieved_ids} | {'✓' if hit else '✗'}\n")

    accuracy = correct / len(test_set)
    print(f"Recall@{k}: {accuracy:.2f} ({correct}/{len(test_set)})")

evaluate_retrieval_v2(test_set, k=3)

Q: How much does the Aurora Sprint weigh?
  Expected: doc_8 | Retrieved: ['doc_8', 'doc_2', 'doc_0'] | ✓

Q: When was Aurora Bike Company founded?
  Expected: doc_0 | Retrieved: ['doc_0', 'doc_13', 'doc_7'] | ✓

Q: How long does the Volt take to charge?
  Expected: doc_11 | Retrieved: ['doc_11', 'doc_5', 'doc_9'] | ✓

Q: What is Aurora's return policy?
  Expected: doc_9 | Retrieved: ['doc_9', 'doc_6', 'doc_3'] | ✓

Q: Where does Aurora manufacture its bikes?
  Expected: doc_12 | Retrieved: ['doc_0', 'doc_12', 'doc_7'] | ✓

Recall@3: 1.00 (5/5)
